# 01 — Repair the coordinate swap (hard G-SWAP)

This notebook repairs the known spider→ant intervention before any science is
run. The workspace band is selected from clean J-Lens visibility only, then a
single fixed token/basis/strength/position configuration is tested on spider
and two predeclared upstream controls. Diagnostics retain the legacy
configuration, alternate token surface, RMS-folded basis, adjacent layer band,
and position-only edits.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['HF_HOME'] = str(Path.home() / '.cache/huggingface')
os.environ['HF_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')

prior = json.loads((ROOT / 'results/metrics.json').read_text())
assert prior['repair_v2']['stage0']['status'] == 'COMPLETE_WITH_RELEASE_OMISSION'
assert prior['repair_v2']['gate_ledger']['g_swap'] in {'UNTESTED', 'PASS', 'FAIL'}

from src.jlens_iface import load_published_lens
from src.model_utils import load_model
from src.v2_repair import MODEL_ID

bundle = load_model(MODEL_ID)
lens = load_published_lens(MODEL_ID)
print({
    'model': bundle.model_id,
    'revision': bundle.revision,
    'dtype': str(next(bundle.hf_model.parameters()).dtype),
    'n_layers': bundle.lens_model.n_layers,
    'lens_source_layers': [min(lens.source_layers), max(lens.source_layers)],
})

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

{'model': 'Qwen/Qwen2.5-7B-Instruct', 'revision': 'a09a35458c702b33eeacc393d103063234e8bc28', 'dtype': 'torch.bfloat16', 'n_layers': 28, 'lens_source_layers': [0, 26]}


## Pre-outcome calibration and repair sweep

The paper's approximately 38–92% depth workspace prior is first mapped to the
28-block model. Inside that prior, the longest contiguous run where the median
minimum source-concept readout rank across three clean prompts is top-10 is
selected. Swap outcomes do not enter band selection. The strict arm uses exact
upstream labels, raw `J.T @ W_U`, `alpha=2`, and all prompt positions.

In [2]:
from src.v2_repair import run_stage1

stage1 = run_stage1(bundle, lens)
print('G1', stage1['g1']['status'], 'max mean KL', stage1['g1']['max_prompt_mean_kl'])
print('workspace prior', stage1['workspace_discovery']['paper_prior_layers'])
print('empirical band', stage1['workspace_discovery']['selected_layers'])
print()
print('Canonical rows:')
for row in stage1['g_swap']['canonical_rows']:
    print({
        'item': row['item_name'],
        'tokens': f"{row['source_surface']!r}->{row['target_surface']!r}",
        'clean_top': row['clean_top_token'],
        'edited_top': row['edited_top_token'],
        'clean_M': row['clean_metric'],
        'edited_M': row['edited_metric'],
        'cf_rank': row['counterfactual_answer_rank_after_edit'],
        'argmax_margin': row['counterfactual_argmax_margin'],
        'repeat_error': row['repeat_max_abs_logit_difference'],
        'pass': row['strict_pass'],
    })
print()
print('G-SWAP', stage1['g_swap']['status'])

G1 PASS max mean KL 1.6602172081547906e-08
workspace prior [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
empirical band [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]

Canonical rows:
{'item': 'spider-legs', 'tokens': "' spider'->'ant'", 'clean_top': '8', 'edited_top': '6', 'clean_M': 6.5, 'edited_M': -6.5, 'cf_rank': 1, 'argmax_margin': 1.125, 'repeat_error': 0.0, 'pass': True}
{'item': 'animal-legs-buffalo2', 'tokens': "' buffalo'->' spider'", 'clean_top': ' four', 'edited_top': ' eight', 'clean_M': 3.5625, 'edited_M': -4.0, 'cf_rank': 1, 'argmax_margin': 2.375, 'repeat_error': 0.0, 'pass': True}
{'item': 'chem-photosynthesis-Z', 'tokens': "' oxygen'->' nitrogen'", 'clean_top': '8', 'edited_top': '7', 'clean_M': 5.375, 'edited_M': -5.25, 'cf_rank': 1, 'argmax_margin': 0.5, 'repeat_error': 0.0, 'pass': True}

G-SWAP PASS


## Persist the gate decision

A PASS licenses only the concept-direction and READ repair notebooks. It does
not license P1–P3. A FAIL would switch the workflow directly to the Stage-4
replication-failure path.

In [3]:
from src.v2_repair import persist_stage1

metrics = persist_stage1(stage1)
gate = metrics['repair_v2']['gate_ledger']['g_swap']
print(json.dumps({
    'G-SWAP': gate,
    'passed': stage1['g_swap']['n_pass'],
    'required': stage1['g_swap']['n_required'],
    'alpha0_no_op_max_error': stage1['alpha0_no_op_max_abs_logit_error'],
    'next': '02_concept_finder' if gate == 'PASS' else '08_report_stage4',
    'science_allowed': False,
}, indent=2))
assert gate == stage1['g_swap']['status']
assert metrics['repair_v2']['gate_ledger']['stage3_science'] == 'PROHIBITED'

{
  "G-SWAP": "PASS",
  "passed": 3,
  "required": 3,
  "alpha0_no_op_max_error": 0.0,
  "next": "02_concept_finder",
  "science_allowed": false
}


In [4]:
import gc
import torch

del stage1, metrics, lens, bundle
gc.collect()
torch.cuda.empty_cache()
print('Notebook 01 complete. Stage-3 science remains prohibited.')

Notebook 01 complete. Stage-3 science remains prohibited.
